# Agent loop using plain Python
### Imports

In [ ]:
import os
import re

from google.colab import userdata
from pathlib import Path

from google import genai
from google.genai import types



### Temporary
This section has code snippets that should be in different modules. When a proper and  python script is developed, this will be moved to appropriate modules and files.


In [ ]:
# Ideally load from file, currently directly as we run on colab
sys_prompt = """
You are a helpful assistant operating in a Reason-Act-Observe loop and expert in a Travel service
Your goal is to analyze a new policy application, analyze it to provide insights and finally validate it. You have access to the following tool:

- insurance_ontology
- insurance_ontology_evaluator
- insurance_validator

You MUST respond using exactly one of these formats:

If you need to use a tool, use this format:
THOUGHT: [Reason about what to do next]
ACTION: [tool_name]: [argument]

If you have the final answer, use this format:
THOUGHT: [Reasoning complete]
FINAL_ANSWER: [Your final response to the user]
"""



os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
client = genai.Client()

# Dummy stock price fetching tool
def get_stock_price(ticker: str) -> str:
    """Retrieves the current stock price for a given ticker symbol."""
    print(f"🔧 [Tool Executing] Fetching price for {ticker.upper()}...")
    mock_database = {"AAPL": "$180.50", "GOOG": "$175.20", "MSFT": "$420.10"}
    return mock_database.get(ticker.upper(), "Ticker not found.")





# Actual Code


# Actual Program


In [ ]:

def run_agentic_loop(user_prompt: str):
    print("Starting agentic loop...")
    print("User entered: ", user_prompt)

    #Get the model
    client = genai.Client()
    model_id = "gemini-2.5-flash"

    # Initialize the conversational context
    messages = [
        types.Content(role="user", parts=[types.Part.from_text(text=user_prompt)])
    ]


    # Bundle the tool reference
    available_tools = {"get_stock_price": get_stock_price}

    max_iterations = 5
    iteration = 0

    while iteration < max_iterations:
        iteration += 1
        print(f"🤖 [Turn {iteration}] Thinking...")

        response = client.models.generate_content(
                model=model_id,
                contents=messages,
                config=types.GenerateContentConfig(
                    tools=[get_stock_price],
                    system_instruction="You are an agentic loop helper. Use your tools whenever a user asks for stock info."
            )
        )

        # Append the assistant's initial response to track the conversation state
        if response.candidates and response.candidates[0].content:
            messages.append(response.candidates[0].content)

        # Check if the model decided to trigger a tool call
        if response.function_calls:
            for call in response.function_calls:
                tool_name = call.name
                tool_args = call.args

                print(f"👉 Model requested tool: {tool_name} with arguments {tool_args}")

                if tool_name in available_tools:
                    # Execute the tool safely using the extracted args
                    tool_output = available_tools[tool_name](**tool_args)
                    print(f"👁️ [Observation] Result: {tool_output}")

                    # Provide the structural execution feedback to the model
                    messages.append(
                        types.Content(
                            role="tool",
                            parts=[
                                types.Part.from_function_response(
                                    name=tool_name,
                                    response={"result": tool_output}
                                )
                            ]
                        )
                    )
                else:
                    print(f"❌ Error: Tool {tool_name} is unregistered.")

            # Continue the loop so Gemini can analyze the newly appended tool result
            continue

        else:
            # If no tool was called, the agent has arrived at its final inference
            print("\n🏁 Final Answer from Gemini:")
            print(response.text)
            break

    else:
        print("⚠️ Closed loop stopped prematurely: Reached max safety iterations.")




if __name__ == "__main__":
    run_agentic_loop("How much is Apple stock right now, and is it higher than Google?")